In [1]:
import warnings
warnings.simplefilter(action='ignore')

import argparse
import os,cv2,math   
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
from torchvision.models.feature_extraction import create_feature_extractor
import torchvision.utils as vutils
from PIL import Image, ImageFilter
from scipy.ndimage import convolve
import os,cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
workers = 2
nz = 100
lr = 0.0001
beta1 = 0.5
ngpu=1
ngf,nc = 3,3
ndf = 64

fsc_submission=numpy.array(pandas.read_csv("/kaggle/input/detect-ai-vs-human-generated-images/test.csv")['id'])
device = torch.device("cuda" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [5]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.swin_v2_b(weights=torchvision.models.swin_transformer.Swin_V2_B_Weights.IMAGENET1K_V1)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1024, 1)
        )
        
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return x

In [6]:
class customiser:
    def __init__(self, batch = 10):
        self.batch = batch
        self.model, self.epoch = 0, 0
        self.data = pandas.read_csv('/kaggle/input/detect-ai-vs-human-generated-images/train.csv')[:40000]
        self.test = pandas.read_csv('/kaggle/input/detect-ai-vs-human-generated-images/test.csv')['id']
        self.train_x, self.train_y = self.data['file_name'], self.data['label']
        
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(self.train_x, 
                                                                                self.train_y, 
                                                                                test_size=0.1, 
                                                                                random_state=100)

        removal_size_train = (len(self.X_train)/10-round(len(self.X_train)/10))*batch
        removal_size_test = (len(self.X_test)/10-round(len(self.X_test)/10))*batch
        if removal_size_train<0:
            removal_size_train = -removal_size_train
        if removal_size_test<0:
            removal_size_test = -removal_size_test
            
        self.X_train, self.X_test, self.y_train, self.y_test = (self.X_train[:round(len(self.X_train)-removal_size_train)], 
                                                                self.X_test[:round(len(self.X_test)-removal_size_test)], 
                                                                self.y_train[:round(len(self.y_train)-removal_size_train)], 
                                                                self.y_test[:round(len(self.y_test)-removal_size_test)])

        self.X_train = numpy.array(self.X_train).reshape(round(len(self.X_train)/10),10)
        self.X_test = numpy.array(self.X_test).reshape(round(len(self.X_test)/10),10)
        self.y_train = numpy.array(self.y_train).reshape(round(len(self.y_train)/10),10)
        self.y_test = numpy.array(self.y_test).reshape(round(len(self.y_test)/10),10)
                
    def image_extracter(self, image_array):
        shape,color=(512,512),cv2.COLOR_BGR2RGB
        processed_image=[]
        
        for i_ in image_array:
            image = cv2.cvtColor(cv2.resize(cv2.imread(f"/kaggle/input/ai-vs-human-generated-dataset/{i_}"), shape), color)
            processed_image.append(image)
        return torch.Tensor(numpy.array(processed_image).reshape((self.batch, 3)+shape)).float().to(device)
        
    def loss_optimizer(self):
        print('loss_optimizer')
        
        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = optim.AdamW(EFF_NET.parameters(), lr=lr, betas=(beta1, 0.999))
        self.scheduler = torch.optim.lr_scheduler.ExponentialLR(self.optimizer, gamma=0.86)
        
    def train(self, EFF_NET):
        print('training')
        accuracy=[]
        i_w, z_k = 0, 0
        
        while True:
            for data, label in zip(self.X_train, self.y_train):
                label = torch.Tensor(label).to(device).float()
                data = customiser.image_extracter(self, data).to(device)
                raw_output = EFF_NET(data).float().view(-1)
                
                err_real = self.criterion(raw_output, label)
                self.optimizer.zero_grad()
                err_real.backward()
                self.optimizer.step()
                
            for data, label in zip(self.X_test, self.y_test):
                label = label
                data = customiser.image_extracter(self, data).to(device)
                raw_output = EFF_NET(data).float().view(-1).detach().cpu().numpy()
                for i in range(len(raw_output)):
                    if raw_output[i]<=0.5:
                        raw_output[i] = 0
                    elif raw_output[i]>=0.5:
                        raw_output[i] = 1
                accuracy.append(accuracy_score(label, raw_output))
            if sum(accuracy)/len(accuracy) >= self.model:
                self.model = sum(accuracy)/len(accuracy)
                self.epoch = z_k
                
            print(f"EPOCH: {z_k} LOSS_FSC: {sum(accuracy)/len(accuracy)}")
            if sum(accuracy)/len(accuracy)>=0.60:
                torch.save(EFF_NET.state_dict(),f'/kaggle/working/{sum(accuracy)/len(accuracy)} {z_k}.pth')# {z_add}{acc}
            if z_k==7:
                break
            z_k+=1
            
    def submission_div(self, EFF_NET):
        sub = ([0]*len(fsc_submission))
        file_ = ['/kaggle/input/detect-ai-vs-human-generated-images-mode-2/0.8924906132665845 1.pth']#f'/kaggle/working/{self.model} {self.epoch}.pth']
        EFF_NET.load_state_dict(torch.load(f"{file_[0]}", weights_only=False,
                                           map_location=torch.device('cpu')))
        for i in range(0, len(fsc_submission), 10):
            img = customiser.image_extracter(self, [fsc_submission[s] for s in range(i, i+10)]).to(device)
            z_arr = EFF_NET(img).sigmoid().cpu().detach().numpy().reshape(-1)
            for j,n in zip(z_arr, range(i, i+10)):
                if j>0.5:
                    sub[n] = 1
                else:
                    sub[n ]= 0
        
        submission=pandas.DataFrame({'id' : fsc_submission, 'label' : numpy.array(sub)})
        pandas.DataFrame(submission).to_csv(f"submission.csv", index=False)
        pandas.DataFrame(submission)

In [7]:
EFF_NET = EffnetModel().float()
EFF_NET = nn.DataParallel(EFF_NET).to(device)

Downloading: "https://download.pytorch.org/models/swin_v2_b-781e5279.pth" to /root/.cache/torch/hub/checkpoints/swin_v2_b-781e5279.pth
100%|██████████| 336M/336M [00:01<00:00, 184MB/s]


In [8]:
cust = customiser()
cust.loss_optimizer()
#cust.train(EFF_NET)
cust.submission_div(EFF_NET)

loss_optimizer
